# DSA4212 — Small LM under a fixed compute budget (Colab runner)

Run top to bottom. Every long step is **resumable** and **skips finished runs**, so if Colab disconnects, reconnect, run sections 1–3 again and re-run the study cell.

| Section | What it does | Approx. time |
|---|---|---|
| 1–3 | GPU check, Drive, code, installs | 2 min |
| 4 | Declared budget (read, don't edit after final runs start) | – |
| 5 | Tokenise WikiText-103 with GPT-2 BPE; cache to Drive | 5–10 min once |
| 6 | Train 4 custom BPE tokenizers (4k/8k/16k/32k) + tokenise | 20–40 min once |
| 7 | Throughput benchmark + 60 s smoke test | 5 min |
| 8 | Studies (tune → final) | budget × #runs |
| 9 | Tables + figures | 1 min |
| 10 | FINAL test-set evaluation (once!) + samples | 2 min |

## 1. GPU

In [ ]:
import torch, subprocess, sys
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Drive + code
Put the `code/` folder in your GitHub repo (`DSA4212_Group_10_Assignment/code/`), push, then this cell clones/pulls it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os
PARQUET = '/content/drive/MyDrive/DSA4212/Dataset'          # the 4 parquet files
DRIVE   = '/content/drive/MyDrive/DSA4212'
RUNS    = f'{DRIVE}/runs'                                    # all results live on Drive
DATA    = '/content/data'                                    # fast local copy of .bin files
TOKS    = f'{DRIVE}/tokenizers'
REPO    = '/content/DSA4212_Group_10_Assignment'
if not Path(REPO).exists():
    !git clone https://github.com/Alutare/DSA4212_Group_10_Assignment.git {REPO}
else:
    !git -C {REPO} pull
CODE = f'{REPO}/code'
assert Path(f'{CODE}/train.py').exists(), "push the code/ folder to GitHub first"
os.chdir(CODE)
for d in (RUNS, DATA, TOKS): Path(d).mkdir(parents=True, exist_ok=True)

## 3. Install

In [ ]:
!pip install -q -r {REPO}/requirements.txt && python test_utils.py

## 4. Declared compute budget  *(copy this into the first page of the report)*
Defined once in `code/experiments.py`.

In [ ]:
import experiments as E
print("Hardware      :", E.HARDWARE)
print("Budget / run  :", E.BUDGET_SECONDS, "s of optimisation-step wall-clock (eval + checkpoint time excluded; compile time included)")
print("Tuning runs   :", round(E.BUDGET_SECONDS*E.TUNE_FRACTION), "s each (preliminary, excluded from the budget)")
print("Init          : random (no pretrained weights/embeddings); tokenizers trained on WikiText train only or GPT-2 BPE merges (no neural components)")
print("Metric        : validation/test bits-per-byte (tokeniser-independent) + perplexity")

## 5. GPT-2 BPE data (cached on Drive after the first time)

In [ ]:
import shutil
def ensure_data(name, tokenizer):
    local, cached = Path(DATA)/name, Path(DRIVE)/'data'/name
    if (local/'meta.json').exists(): return print(name, 'ready (local)')
    if (cached/'meta.json').exists():
        shutil.copytree(cached, local); return print(name, 'copied from Drive')
    !python prepare_data.py --data_dir {PARQUET} --tokenizer {tokenizer} --name {name} --out_root {DATA}
    shutil.copytree(local, cached, dirs_exist_ok=True)
ensure_data('gpt2', 'gpt2')

## 6. Custom tokenizers (trained on WikiText **train** only; not part of the run budget)

In [ ]:
for V, name in [(4096,'bpe4k'), (8192,'bpe8k'), (16384,'bpe16k'), (32768,'bpe32k')]:
    path = f'{TOKS}/{name}.json'
    if not Path(path).exists():
        !python train_tokenizer.py --data_dir {PARQUET} --vocab_size {V} --out {path}
    ensure_data(name, path)

## 7. Benchmark + smoke test

In [ ]:
!python benchmark.py --sizes xs s m l xl --vocabs 50257 16384 --out {RUNS}/benchmark.csv

In [ ]:
!python run_experiments.py --study size --stage smoke --only size_xs --tok gpt2 --data_root {DATA} --out_root {RUNS} --budget 60

## 8. Studies
Recommended order. **Tune** runs (5 min) pick hyper-parameters on *validation*; **final** runs use the full declared budget.

1. `tokenizer` (final) → pick the best tokenizer by validation BPB → set `TOK`.
2. `lr` (tune) → update `lr` in `SIZES` if a different value clearly wins.
3. `size` (final) ← **your headline question**
4. `shape`, `efficiency` (final)
5. `seeds` (final, 3 seeds of your best config) → error bars.
6. optional: `context`, `batch`, `data`

In [ ]:
TOK = 'gpt2'    # <- change after the tokenizer study
SIZE = 'm'
STUDY_ARGS = f"--data_root {DATA} --out_root {RUNS} --tok {TOK} --size {SIZE}"

In [ ]:
!python run_experiments.py --study tokenizer --stage final --data_root {DATA} --out_root {RUNS}

In [ ]:
!python run_experiments.py --study lr --stage tune {STUDY_ARGS}

In [ ]:
!python run_experiments.py --study size --stage final {STUDY_ARGS}

In [ ]:
!python run_experiments.py --study shape --stage final {STUDY_ARGS}

In [ ]:
!python run_experiments.py --study efficiency --stage final {STUDY_ARGS}

In [ ]:
!python run_experiments.py --study seeds --stage final --seeds 1337 1338 1339 {STUDY_ARGS}

In [ ]:
# optional
!python run_experiments.py --study context --stage final {STUDY_ARGS}
!python run_experiments.py --study batch --stage final {STUDY_ARGS}
!python run_experiments.py --study data --stage final {STUDY_ARGS}

## 9. Tables and figures

In [ ]:
!python analyze.py --root {RUNS} --stage final --out {DRIVE}/figures
from IPython.display import Image, display
for f in sorted(Path(f'{DRIVE}/figures').glob('*_summary.png')): display(Image(str(f)))

## 10. FINAL test evaluation  *(touch the test set only for your chosen final configurations, once each)*
Edit `FINAL_RUNS` to the run directories you report. `evaluate.py` refuses to run on test without `--final`, and refuses a second test evaluation of the same run.

In [ ]:
FINAL_RUNS = [f'{RUNS}/final/size/size_m_{TOK}']      # <- list the runs you report
for r in FINAL_RUNS:
    !python evaluate.py --ckpt {r}/ckpt.pt --data_dir {DATA}/{TOK} --split test --final

In [ ]:
!python sample.py --ckpt {FINAL_RUNS[0]}/ckpt.pt --data_dir {DATA}/{TOK} --tokens 150